# trapX LLM Advisory — Dry-Run for Bars **11:06 & 11:07** (2026-06-05)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chanakya1998begin/rdp/blob/main/dry_run_1106_1107_colab.ipynb)
### Colab runner · NVIDIA DGX Cloud (`meta/llama-3.3-70b-instruct`)

This notebook re-runs the per-bar **LLM advisory specialist panel** for the two bars, end-to-end, **inside Colab** — no DB, no session log, no repo clone, no Google Drive mount required.

**Touchpoints replayed**

| Bar | Touchpoints |
|-----|-------------|
| 11:06 | `jerk_drilldown`, `big_volume_1m` |
| 11:07 | `jerk_drilldown` |

**How it works**
1. The full self-contained script `dry_run_1106_1107.py` is written to `/content/` (cell 2). It already embeds every skill (system prompt) and the captured verdicts, so it runs on plain Python.
2. The **input data points** for each touchpoint are written as editable JSON files under `/content/inputs/` (cell 3). The script reads these by default — *edit a file, re-run, and the verdict changes*.
3. Cell 4 runs the panel **live** against NVIDIA (falls back to the captured replay if the key/SDK is missing).

**Modes**
- *auto* (default): live NVIDIA call if `NVIDIA_API_KEY` + `openai` are present, else offline replay.
- `--replay`: force offline replay of captured responses (no key needed).
- `--live`: force a fresh NVIDIA call (`temperature=0.0`, `seed=42`, matches the engine).
- `--no-input-files`: ignore `/content/inputs/*.snapshot.json` and use the embedded bundle.


## 1. NVIDIA key + OpenAI SDK
Add your key in **Colab Secrets** (🔑 lock icon → *Add new secret* → name `NVIDIA_API_KEY`, value `nvapi-...` → enable for this notebook). If no secret is found, you'll be prompted to paste it.


In [ ]:
# 1.1 install OpenAI SDK + load NVIDIA_API_KEY (Colab Secret -> getpass fallback)
!pip install -q openai

import os
key = None
try:
    from google.colab import userdata
    key = userdata.get('NVIDIA_API_KEY')
except Exception:
    pass
if not key:
    import getpass
    key = getpass.getpass('NVIDIA_API_KEY (nvapi-...): ')

os.environ['NVIDIA_API_KEY'] = (key or '').strip()
assert os.environ['NVIDIA_API_KEY'], "NVIDIA_API_KEY is empty — set it in Colab Secrets or paste it above."

# Smoke ping — verify the key works and the gateway is reachable
from openai import OpenAI
_c = OpenAI(base_url='https://integrate.api.nvidia.com/v1',
            api_key=os.environ['NVIDIA_API_KEY'], timeout=30)
_p = _c.chat.completions.create(
    model='meta/llama-3.3-70b-instruct',
    messages=[{'role': 'user', 'content': "Say 'ok' and stop."}],
    max_tokens=8, temperature=0.0,
)
print('NVIDIA gateway reachable. ping:', (_p.choices[0].message.content or '').strip())
print('  tokens in/out:', _p.usage.prompt_tokens, '/', _p.usage.completion_tokens)


## 2. Write the self-contained dry-run script
This is `project/llm_advisory/dry_run_1106_1107.py` verbatim. It embeds the skills (system prompts) and the captured responses, so nothing else is needed to run it.


In [ ]:
%%writefile /content/dry_run_1106_1107.py
"""Standalone LLM-advisory dry-run for 2026-06-05 bars 11:06 and 11:07.

SELF-CONTAINED. The captured LLM requests (system skill + user snapshot)
and the original verdicts for every touchpoint that fired on these two
bars are embedded below as a base64 JSON bundle. No DB, no session log,
no trapx_agent import is required, so this file runs on any machine with
plain Python 3.

Touchpoints replayed (the per-bar "specialist" panel):
  bar 11:06 : jerk_drilldown, big_volume_1m
  bar 11:07 : jerk_drilldown

Modes:
  (default) auto : call NVIDIA live if NVIDIA_API_KEY + openai SDK are
                   present, otherwise fall back to REPLAY of the captured
                   responses (no network, stdlib only).
  --live         : force a live NVIDIA call (matches the engine exactly:
                   temperature=0.0, seed=42).
  --replay       : force replay of the captured responses (offline).

Every run writes a forensic log next to this script at
reports/2026-06-05-bars-1106-1107-dryrun.log and prints a per-bar
verdict board to stdout.

Usage:
    python dry_run_1106_1107.py            # auto (live if key present)
    python dry_run_1106_1107.py --replay   # offline, no key needed
    python dry_run_1106_1107.py --live     # force a fresh NVIDIA call
"""
from __future__ import annotations

import argparse
import base64
import json
import os
import re
import sys
from datetime import datetime
from pathlib import Path

# Windows cp1252 console cannot encode the unicode glyphs we print.
for _s in (sys.stdout, sys.stderr):
    try:
        _s.reconfigure(encoding="utf-8")  # type: ignore[attr-defined]
    except Exception:
        pass

# ── embedded captured bundle (base64 of UTF-8 JSON) ─────────────────────────
BUNDLE_B64 = "..."

NVIDIA_BASE_URL    = "https://integrate.api.nvidia.com/v1"
NVIDIA_TEMPERATURE = 0.0
NVIDIA_SEED        = 42

SCRIPT_DIR = Path(__file__).resolve().parent
DATE_LABEL = "2026-06-05"
REPORT_DIR = SCRIPT_DIR / "reports"
TRACE_LOG  = REPORT_DIR / (DATE_LABEL + "-bars-1106-1107-dryrun.log")

KNOWN_EMOJI = "✅⚠️❌\U0001f504\U0001f7e2\U0001f534\U0001f7e1⚪\U0001f535\U0001f7e0"


class _Tee:
    """Tee logger: terse to stdout, verbose to the forensic trace file."""

    def __init__(self, path):
        path.parent.mkdir(parents=True, exist_ok=True)
        self._fh = path.open("w", encoding="utf-8")
        self._t0 = datetime.now()
        self._fh.write(
            "# LLM-advisory dry-run - bars 11:06 and 11:07 on " + DATE_LABEL + "\n"
            "# Run started: " + self._t0.isoformat() + "\n\n"
        )
        self._fh.flush()

    def log(self, msg, prefix="    "):
        ts = (datetime.now() - self._t0).total_seconds()
        self._fh.write("[%7.3fs]%s%s\n" % (ts, prefix, msg))
        self._fh.flush()

    def section(self, title):
        bar = "=" * 60
        self._fh.write("\n" + bar + "\n  " + title + "\n" + bar + "\n")
        self._fh.flush()
        print("\n▸ " + title)

    def step(self, msg):
        print("  · " + msg)
        self.log(msg)

    def close(self):
        self._fh.write("\n# Run finished: " + datetime.now().isoformat() + "\n")
        self._fh.close()


def _find_env():
    p = SCRIPT_DIR
    for _ in range(6):
        c = p / ".env"
        if c.exists():
            return c
        if p.parent == p:
            break
        p = p.parent
    return None


def _load_env(trace):
    path = _find_env()
    if not path:
        trace.log(".env not found in parent dirs - relying on existing env vars")
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        v = v.strip().strip('"').strip("'")
        os.environ.setdefault(k.strip(), v)
    trace.log(".env loaded from " + str(path))


def _parse_verdict(text):
    """Best-effort extraction of (verdict_line, score, action_line)."""
    verdict_line = ""
    score = None
    action = ""
    lines = [ln.rstrip() for ln in text.splitlines()]
    for ln in lines:
        if ln.strip() and any(e in ln for e in KNOWN_EMOJI):
            verdict_line = ln.strip()
            break
    m = re.search(r"[Ss]core:?\s*\[?\s*([+-]?\d+(?:\.\d+)?)", text)
    if not m:
        m = re.search(r"\[\s*([+-]?\d+(?:\.\d+)?)\s*\]", text)
    if m:
        score = float(m.group(1))
    for ln in lines:
        if ln.strip().lower().startswith("action"):
            action = ln.strip()
            break
    return verdict_line, score, action


def _extract_snapshot(user_message):
    """Pull the JSON snapshot (the input data points) out of the captured
    user message. The message format is '...prose...\nSnapshot:\n{...}'."""
    idx = user_message.find("{")
    if idx < 0:
        return None
    try:
        return json.loads(user_message[idx:])
    except Exception:
        return None


def _resolve_user_message(r, use_files):
    """Decide which inputs feed this touchpoint's verdict.

    If use_files is True and inputs/<DATE>_<bar>_<touchpoint>.snapshot.json
    exists next to this script, load THAT snapshot (so an edit to the file
    changes the run) and rebuild the user message around it. Otherwise use
    the snapshot embedded in this self-contained file.

    Returns (user_message, source_label).
    """
    embedded = r["user_message"]
    if not use_files:
        return embedded, "embedded bundle"
    name = "%s_%s_%s.snapshot.json" % (
        DATE_LABEL, r["bar_time"].replace(":", ""), r["touchpoint"])
    path = SCRIPT_DIR / "inputs" / name
    if not path.exists():
        return embedded, "embedded bundle (no inputs file: %s)" % name
    try:
        snap = json.loads(path.read_text(encoding="utf-8"))
    except Exception as e:
        return embedded, "embedded bundle (inputs file parse error: %s)" % e
    cut = embedded.find("Snapshot:")
    prefix = embedded[:cut] if cut >= 0 else (embedded.split("{", 1)[0])
    user_message = prefix + "Snapshot:\n" + json.dumps(snap, indent=2, ensure_ascii=False)
    return user_message, "inputs file: %s" % name


def _flatten(obj, prefix=""):
    """Flatten a nested snapshot into (dotted_path, value) leaf rows so every
    input data point is visible on its own line for cross-checking."""
    rows = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            p = (prefix + "." + str(k)) if prefix else str(k)
            rows.extend(_flatten(v, p))
    elif isinstance(obj, list):
        if not obj:
            rows.append((prefix, "[]"))
        elif all(not isinstance(x, (dict, list)) for x in obj):
            rows.append((prefix, obj))          # scalar list -> one row
        else:
            for i, v in enumerate(obj):
                rows.extend(_flatten(v, prefix + "[" + str(i) + "]"))
    else:
        rows.append((prefix, obj))
    return rows


def _print_input_data_points(r, trace):
    """Render every input data point fed to this touchpoint's verdict."""
    tp, bar = r["touchpoint"], r["bar_time"]
    snap = _extract_snapshot(r["user_message"])
    print("")
    print("  --- INPUT DATA POINTS: %s @ %s ---" % (tp, bar))
    if snap is None:
        print("  (could not parse snapshot; raw user_message follows)")
        print("  " + r["user_message"][:500])
        trace.log("snapshot parse failed for %s @ %s" % (tp, bar))
        return
    rows = _flatten(snap)
    for path, val in rows:
        print("     %-44s = %s" % (path, val))
    print("  (%d input data points)" % len(rows))
    trace.log("input snapshot for %s @ %s (%d leaf data points):" % (tp, bar, len(rows)))
    trace.log(json.dumps(snap, indent=2, ensure_ascii=False))


def _call_nvidia(system, user, model, trace):
    from openai import OpenAI
    key = os.environ.get("NVIDIA_API_KEY", "").strip()
    if not key:
        raise RuntimeError("NVIDIA_API_KEY not set in environment")
    client = OpenAI(base_url=NVIDIA_BASE_URL, api_key=key, timeout=60)
    t0 = datetime.now()
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=NVIDIA_TEMPERATURE,
        seed=NVIDIA_SEED,
    )
    dt = (datetime.now() - t0).total_seconds()
    text = (resp.choices[0].message.content or "").strip()
    trace.log("NVIDIA ok in %.2fs  model=%s  out_chars=%d" % (dt, model, len(text)))
    return text, dt


def main():
    ap = argparse.ArgumentParser(description="Dry-run LLM advisory for bars 11:06 / 11:07")
    g = ap.add_mutually_exclusive_group()
    g.add_argument("--live", action="store_true", help="force a live NVIDIA call")
    g.add_argument("--replay", action="store_true", help="force replay of captured responses")
    ap.add_argument("--no-input-files", action="store_true",
                    help="ignore inputs/*.snapshot.json; use the embedded bundle only")
    args = ap.parse_args()

    bundle = json.loads(base64.b64decode(BUNDLE_B64).decode("utf-8"))
    records = bundle["records"]

    trace = _Tee(TRACE_LOG)
    try:
        trace.section("Phase 0 - Setup")
        _load_env(trace)
        have_key = bool(os.environ.get("NVIDIA_API_KEY", "").strip())
        try:
            import openai  # noqa: F401
            have_openai = True
        except Exception:
            have_openai = False

        if args.replay:
            mode = "replay"
        elif args.live:
            mode = "live"
        else:
            mode = "live" if (have_key and have_openai) else "replay"
        trace.step("mode = %s   (NVIDIA_API_KEY=%s, openai_sdk=%s)" % (mode, have_key, have_openai))
        trace.step("records embedded: %d" % len(records))

        bars = []
        for r in records:
            if r["bar_time"] not in bars:
                bars.append(r["bar_time"])

        board = {}
        for bar in bars:
            trace.section("BAR " + bar)
            board[bar] = []
            for r in records:
                if r["bar_time"] != bar:
                    continue
                tp = r["touchpoint"]
                trace.step("touchpoint %s  (skill=%s, rules_sha=%s)" % (tp, r["skill_file"], r.get("rules_sha")))

                # Choose inputs: on-disk inputs/*.snapshot.json (editable) or
                # the embedded bundle. This is what makes the dry-run actually
                # run "from the input files".
                user_message, in_src = _resolve_user_message(r, not args.no_input_files)
                r = dict(r)
                r["user_message"] = user_message
                trace.step("input source: " + in_src)
                trace.log("  user_message chars=%d, system chars=%d" % (len(user_message), len(r["system_prompt"])))

                # Surface every input data point that fed this verdict.
                _print_input_data_points(r, trace)

                if mode == "live":
                    try:
                        text, dt = _call_nvidia(r["system_prompt"], r["user_message"], r["model"], trace)
                        src = "LIVE NVIDIA"
                    except Exception as e:
                        trace.step("live failed (%s: %s) -> replay for this touchpoint" % (type(e).__name__, e))
                        text, dt, src = r["captured_raw_text"], 0.0, "REPLAY (captured)"
                else:
                    text, dt, src = r["captured_raw_text"], 0.0, "REPLAY (captured)"

                verdict, score, action = _parse_verdict(text)
                board[bar].append((tp, verdict or "(no verdict line)", score))

                print("")
                print("  +---- \U0001f916 %s @ %s   [%s] " % (tp, bar, src) + "-" * 18)
                for ln in (text or "(empty)").strip().splitlines():
                    print("  | " + ln)
                print("  +- parsed score=%s   |   original captured: verdict=%r score=%s"
                      % (score, r.get("captured_verdict"), r.get("captured_score")))
                trace.log("rendered %s @ %s  src=%s latency=%.2fs score=%s" % (tp, bar, src, dt, score))
                trace.log("  raw_text:\n" + (text or ""))

        trace.section("PER-BAR VERDICT BOARD (isolated touchpoints)")
        print("")
        print("  ================ PER-BAR VERDICT BOARD ================")
        for bar in bars:
            print("  BAR %s:" % bar)
            for tp, verdict, score in board[bar]:
                print("     - %-16s score=%-6s %s" % (tp, str(score), verdict))
            scores = [s for _, _, s in board[bar] if isinstance(s, (int, float))]
            if scores:
                avg = sum(scores) / len(scores)
                print("     naive mean of touchpoint scores = %+.3f" % avg)
                print("     ^ NOTE: a naive mean is NOT a negotiated verdict - see the")
                print("       attending-physician synthesis discussion. This board shows")
                print("       why isolated touchpoints need one consultant to reconcile them.")
        print("")
        print("  Full forensic log: " + str(TRACE_LOG))
        return 0
    finally:
        trace.close()


if __name__ == "__main__":
    raise SystemExit(main())


## 3. Input data files (editable)
Each file below is the **exact input snapshot** (the data points) fed to one touchpoint's verdict on that bar. They are written to `/content/inputs/` and the script reads them by default — **edit any value, re-run cell 4, and the verdict updates.**

| File | Touchpoint | Bar |
|------|-----------|-----|
| `2026-06-05_1106_jerk_drilldown.snapshot.json` | jerk_drilldown | 11:06 |
| `2026-06-05_1106_big_volume_1m.snapshot.json` | big_volume_1m | 11:06 |
| `2026-06-05_1107_jerk_drilldown.snapshot.json` | jerk_drilldown | 11:07 |


In [ ]:
import os
os.makedirs('/content/inputs', exist_ok=True)
print('inputs dir ready: /content/inputs')

In [ ]:
%%writefile /content/inputs/2026-06-05_1106_jerk_drilldown.snapshot.json
{
  "bar_time": "11:06",
  "jerk_pct": 9.06505163513585,
  "jerk_dir": "UP",
  "stack_depth": 36,
  "prior_3bar_jerks": [
    -12.56,
    -19.82,
    -35.19
  ],
  "jerk_event_kind": "sustained",
  "sniper": {
    "bias": "UP",
    "glyph": "↑",
    "headline": "UP CONFIRMED · UP LEAN · ceiling weak (jerk agrees)",
    "action": "UP · cautious",
    "pe_state": "MIXED",
    "ce_state": "UNWOUND",
    "tail_share": 20.831284572833436
  },
  "writer_contribution": {
    "trn_delta": 4230200,
    "ALL_PE_signed": 314340,
    "ALL_CE_signed": -3915860,
    "ALL_PE_pct": 7.43,
    "ALL_CE_pct": -92.57,
    "HIGH_DELTA_PE_signed": 107380,
    "HIGH_DELTA_CE_signed": -820365,
    "HIGH_DELTA_PE_pct": 2.54,
    "HIGH_DELTA_CE_pct": -19.39,
    "pro_share": 2.54,
    "pro_role": "PE",
    "regime": "CAPITULATION-LED · pros absent"
  },
  "top3_by_side": {
    "aligned_top3": [
      {
        "strike": 23500,
        "typ": "PE",
        "wgt": 0.5,
        "delta": 236600
      },
      {
        "strike": 23350,
        "typ": "PE",
        "wgt": 0.2,
        "delta": -164840
      },
      {
        "strike": 23450,
        "typ": "PE",
        "wgt": 0.4,
        "delta": 116350
      }
    ],
    "counter_top3": [
      {
        "strike": 23800,
        "typ": "CE",
        "wgt": 0.0,
        "delta": -570700
      },
      {
        "strike": 23600,
        "typ": "CE",
        "wgt": 0.2,
        "delta": -558805
      },
      {
        "strike": 23500,
        "typ": "CE",
        "wgt": 0.4,
        "delta": -462930
      }
    ]
  },
  "flow_decomposition": {
    "PE_fresh_writes": [
      {
        "strike": 23500,
        "wgt": 0.5,
        "delta": 236600
      },
      {
        "strike": 23450,
        "wgt": 0.4,
        "delta": 116350
      },
      {
        "strike": 23300,
        "wgt": 0.1,
        "delta": 90285
      },
      {
        "strike": 23600,
        "wgt": 0.7,
        "delta": 73125
      },
      {
        "strike": 23650,
        "wgt": 0.8,
        "delta": 57460
      },
      {
        "strike": 23550,
        "wgt": 0.6,
        "delta": 51805
      },
      {
        "strike": 23400,
        "wgt": 0.3,
        "delta": 29900
      },
      {
        "strike": 23100,
        "wgt": 0.0,
        "delta": 16315
      },
      {
        "strike": 23800,
        "wgt": 1.0,
        "delta": 5720
      },
      {
        "strike": 23700,
        "wgt": 0.9,
        "delta": 325
      }
    ],
    "PE_unwinds": [
      {
        "strike": 23350,
        "wgt": 0.2,
        "delta": -164840
      },
      {
        "strike": 23750,
        "wgt": 1.0,
        "delta": -81055
      },
      {
        "strike": 23150,
        "wgt": 0.0,
        "delta": -50245
      },
      {
        "strike": 23250,
        "wgt": 0.0,
        "delta": -35555
      },
      {
        "strike": 23200,
        "wgt": 0.0,
        "delta": -31850
      }
    ],
    "CE_fresh_writes": [],
    "CE_unwinds": [
      {
        "strike": 23800,
        "wgt": 0.0,
        "delta": -570700
      },
      {
        "strike": 23600,
        "wgt": 0.2,
        "delta": -558805
      },
      {
        "strike": 23500,
        "wgt": 0.4,
        "delta": -462930
      },
      {
        "strike": 23650,
        "wgt": 0.1,
        "delta": -410735
      },
      {
        "strike": 23400,
        "wgt": 0.6,
        "delta": -408070
      },
      {
        "strike": 23450,
        "wgt": 0.5,
        "delta": -406510
      },
      {
        "strike": 23700,
        "wgt": 0.0,
        "delta": -331825
      },
      {
        "strike": 23550,
        "wgt": 0.3,
        "delta": -273975
      },
      {
        "strike": 23350,
        "wgt": 0.7,
        "delta": -163085
      },
      {
        "strike": 23300,
        "wgt": 0.8,
        "delta": -143130
      },
      {
        "strike": 23750,
        "wgt": 0.0,
        "delta": -80015
      },
      {
        "strike": 23200,
        "wgt": 1.0,
        "delta": -72995
      },
      {
        "strike": 23250,
        "wgt": 0.9,
        "delta": -23270
      },
      {
        "strike": 23100,
        "wgt": 1.0,
        "delta": -7670
      },
      {
        "strike": 23150,
        "wgt": 1.0,
        "delta": -2145
      }
    ],
    "PE_fresh_pct": 16.02,
    "PE_unwind_pct": -8.59,
    "CE_fresh_pct": 0.0,
    "CE_unwind_pct": -92.57
  },
  "near_money_zone": {
    "PE_near_money_strikes": [
      {
        "strike": 23500,
        "wgt": 0.5,
        "delta": 236600
      },
      {
        "strike": 23450,
        "wgt": 0.4,
        "delta": 116350
      },
      {
        "strike": 23400,
        "wgt": 0.3,
        "delta": 29900
      }
    ],
    "CE_near_money_strikes": [],
    "PE_near_money_pct": 9.05,
    "CE_near_money_pct": 0.0
  },
  "straddle_candidates": [
    {
      "strike": 23350,
      "ce_delta": -163085,
      "pe_delta": -164840,
      "kind": "both_unwound"
    },
    {
      "strike": 23750,
      "ce_delta": -80015,
      "pe_delta": -81055,
      "kind": "both_unwound"
    }
  ],
  "structural_levels": {
    "PE_floor_band": {
      "low": 23450,
      "high": 23500
    },
    "CE_ceiling_band": null
  },
  "per_bar": {
    "signal": 0.0,
    "premium": 81.25,
    "atr": 13.49,
    "twap": 0.0,
    "twap_stretch_atr": null,
    "spot": 23486.95,
    "fut": 23568.2
  }
}

In [ ]:
%%writefile /content/inputs/2026-06-05_1106_big_volume_1m.snapshot.json
{
  "kind": "1m",
  "direction": "UP",
  "window_start": "11:06",
  "bar_time": "11:06",
  "vol_pts": 127010,
  "vol_threshold": 125000,
  "vol_ratio": 1.01608,
  "body_size_pts": 21.200000000000728,
  "body_pct": 61.80758017493054,
  "source": "FUT",
  "signal_now": 0.0,
  "running_atr": 13.49,
  "regime": null,
  "is_near_lis": false,
  "prior_3bar_vol_ratios": null
}

In [ ]:
%%writefile /content/inputs/2026-06-05_1107_jerk_drilldown.snapshot.json
{
  "bar_time": "11:07",
  "jerk_pct": 12.89764069077142,
  "jerk_dir": "UP",
  "stack_depth": 37,
  "prior_3bar_jerks": [
    9.07,
    -12.56,
    -19.82
  ],
  "jerk_event_kind": "sustained",
  "sniper": {
    "bias": "UP",
    "glyph": "↑",
    "headline": "UP CONFIRMED · UP LEAN · ceiling weak (jerk agrees)",
    "action": "UP · cautious",
    "pe_state": "MIXED",
    "ce_state": "UNWOUND",
    "tail_share": 11.581618877909177
  },
  "writer_contribution": {
    "trn_delta": 6018675,
    "ALL_PE_signed": 1138475,
    "ALL_CE_signed": -4880200,
    "ALL_PE_pct": 18.92,
    "ALL_CE_pct": -81.08,
    "HIGH_DELTA_PE_signed": 339365,
    "HIGH_DELTA_CE_signed": -1387165,
    "HIGH_DELTA_PE_pct": 5.64,
    "HIGH_DELTA_CE_pct": -23.05,
    "pro_share": 5.64,
    "pro_role": "PE",
    "regime": "CAPITULATION-LED · pros absent"
  },
  "top3_by_side": {
    "aligned_top3": [
      {
        "strike": 23500,
        "typ": "PE",
        "wgt": 0.5,
        "delta": 541840
      },
      {
        "strike": 23450,
        "typ": "PE",
        "wgt": 0.4,
        "delta": 252395
      },
      {
        "strike": 23400,
        "typ": "PE",
        "wgt": 0.3,
        "delta": 175110
      }
    ],
    "counter_top3": [
      {
        "strike": 23400,
        "typ": "CE",
        "wgt": 0.6,
        "delta": -911820
      },
      {
        "strike": 23500,
        "typ": "CE",
        "wgt": 0.4,
        "delta": -713245
      },
      {
        "strike": 23700,
        "typ": "CE",
        "wgt": 0.0,
        "delta": -685360
      }
    ]
  },
  "flow_decomposition": {
    "PE_fresh_writes": [
      {
        "strike": 23500,
        "wgt": 0.5,
        "delta": 541840
      },
      {
        "strike": 23450,
        "wgt": 0.4,
        "delta": 252395
      },
      {
        "strike": 23400,
        "wgt": 0.3,
        "delta": 175110
      },
      {
        "strike": 23350,
        "wgt": 0.2,
        "delta": 132275
      },
      {
        "strike": 23550,
        "wgt": 0.6,
        "delta": 123435
      },
      {
        "strike": 23600,
        "wgt": 0.7,
        "delta": 122980
      },
      {
        "strike": 23650,
        "wgt": 0.8,
        "delta": 52845
      },
      {
        "strike": 23700,
        "wgt": 0.9,
        "delta": 49270
      }
    ],
    "PE_unwinds": [
      {
        "strike": 23200,
        "wgt": 0.0,
        "delta": -105365
      },
      {
        "strike": 23300,
        "wgt": 0.1,
        "delta": -68640
      },
      {
        "strike": 23100,
        "wgt": 0.0,
        "delta": -55640
      },
      {
        "strike": 23150,
        "wgt": 0.0,
        "delta": -39390
      },
      {
        "strike": 23250,
        "wgt": 0.0,
        "delta": -33475
      },
      {
        "strike": 23750,
        "wgt": 1.0,
        "delta": -9165
      }
    ],
    "CE_fresh_writes": [
      {
        "strike": 23750,
        "wgt": 0.0,
        "delta": 10205
      }
    ],
    "CE_unwinds": [
      {
        "strike": 23400,
        "wgt": 0.6,
        "delta": -911820
      },
      {
        "strike": 23500,
        "wgt": 0.4,
        "delta": -713245
      },
      {
        "strike": 23700,
        "wgt": 0.0,
        "delta": -685360
      },
      {
        "strike": 23450,
        "wgt": 0.5,
        "delta": -622895
      },
      {
        "strike": 23600,
        "wgt": 0.2,
        "delta": -614510
      },
      {
        "strike": 23550,
        "wgt": 0.3,
        "delta": -332150
      },
      {
        "strike": 23650,
        "wgt": 0.1,
        "delta": -279305
      },
      {
        "strike": 23800,
        "wgt": 0.0,
        "delta": -255775
      },
      {
        "strike": 23300,
        "wgt": 0.8,
        "delta": -238030
      },
      {
        "strike": 23350,
        "wgt": 0.7,
        "delta": -153205
      },
      {
        "strike": 23250,
        "wgt": 0.9,
        "delta": -33800
      },
      {
        "strike": 23200,
        "wgt": 1.0,
        "delta": -30420
      },
      {
        "strike": 23100,
        "wgt": 1.0,
        "delta": -13520
      },
      {
        "strike": 23150,
        "wgt": 1.0,
        "delta": -6370
      }
    ],
    "PE_fresh_pct": 24.09,
    "PE_unwind_pct": -5.18,
    "CE_fresh_pct": 0.17,
    "CE_unwind_pct": -81.25
  },
  "near_money_zone": {
    "PE_near_money_strikes": [
      {
        "strike": 23500,
        "wgt": 0.5,
        "delta": 541840
      },
      {
        "strike": 23450,
        "wgt": 0.4,
        "delta": 252395
      },
      {
        "strike": 23400,
        "wgt": 0.3,
        "delta": 175110
      }
    ],
    "CE_near_money_strikes": [],
    "PE_near_money_pct": 16.11,
    "CE_near_money_pct": 0.0
  },
  "straddle_candidates": [
    {
      "strike": 23300,
      "ce_delta": -238030,
      "pe_delta": -68640,
      "kind": "both_unwound"
    }
  ],
  "structural_levels": {
    "PE_floor_band": {
      "low": 23400,
      "high": 23500
    },
    "CE_ceiling_band": null
  },
  "per_bar": {
    "signal": 0.0,
    "premium": 85.15000000000146,
    "atr": 14.1,
    "twap": 0.0,
    "twap_stretch_atr": null,
    "spot": 23482.05,
    "fut": 23567.2
  }
}

## 4. Run the panel
Auto mode → **live NVIDIA** because the key is set above. The script prints, per bar: every input data point, each touchpoint's rendered verdict, and a per-bar verdict board.


In [ ]:
!cd /content && python dry_run_1106_1107.py

#### Offline replay (no API call) — uses the captured responses

In [ ]:
!cd /content && python dry_run_1106_1107.py --replay

## 5. Full forensic log

In [ ]:
log = '/content/reports/2026-06-05-bars-1106-1107-dryrun.log'
print(open(log, encoding='utf-8').read())
